<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/MOEA_D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# MOEA/D MULTI-OBJECTIVE QUANT PORTFOLIO OPTIMISATION
# Return ↑ | CVaR ↓ | Turnover ↓ | Diversification ↑
# ================================================================

!pip -q install yfinance pymoo

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.moead import MOEAD
from pymoo.util.ref_dirs import get_reference_directions
from pymoo.optimize import minimize

np.random.seed(42)

# ================================================================
# 1. DATA
# ================================================================

tickers = [
    "RELIANCE.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "INFY.NS",
    "TCS.NS",
    "ITC.NS",
    "LT.NS",
    "SBIN.NS",
    "BHARTIARTL.NS",
    "HINDUNILVR.NS"
]

prices = yf.download(
    tickers,
    start="2021-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

returns = prices.pct_change().dropna()

n_assets = len(tickers)

# Annualised expected return
mu = returns.mean().values * 252

# Annualised covariance
cov = returns.cov().values * 252

# Equal-weight reference portfolio
reference = np.ones(n_assets) / n_assets


# ================================================================
# 2. PORTFOLIO METRICS
# ================================================================

def metrics(w):

    daily = returns.values @ w

    # Expected annual return
    annual_return = np.dot(w, mu)

    # Annual volatility
    volatility = np.sqrt(
        w @ cov @ w
    )

    # ------------------------------------------------------------
    # 95% CVaR
    # ------------------------------------------------------------

    var95 = np.percentile(
        daily,
        5
    )

    tail = daily[daily <= var95]

    cvar = (
        -tail.mean() * np.sqrt(252)
        if len(tail) > 0
        else 0
    )

    # ------------------------------------------------------------
    # Turnover
    # ------------------------------------------------------------

    turnover = np.sum(
        np.abs(w - reference)
    )

    # ------------------------------------------------------------
    # Diversification
    # ------------------------------------------------------------

    hhi = np.sum(w ** 2)

    diversification = 1 / hhi

    # ------------------------------------------------------------
    # Sharpe
    # ------------------------------------------------------------

    sharpe = (
        annual_return / volatility
        if volatility > 0
        else 0
    )

    return (
        annual_return,
        volatility,
        cvar,
        turnover,
        diversification,
        sharpe
    )


# ================================================================
# 3. MOEA/D PORTFOLIO PROBLEM
# ================================================================
# IMPORTANT:
# MOEA/D implementation in pymoo does not support explicit
# constraints. Therefore weights are normalised internally.
# ================================================================

class PortfolioProblem(ElementwiseProblem):

    def __init__(self):

        super().__init__(
            n_var=n_assets,
            n_obj=4,
            xl=np.zeros(n_assets),
            xu=np.ones(n_assets)
        )

    def _evaluate(
        self,
        w,
        out,
        *args,
        **kwargs
    ):

        # Long-only
        w = np.maximum(
            w,
            0
        )

        # Fully invested
        if w.sum() == 0:

            w = (
                np.ones(n_assets)
                / n_assets
            )

        else:

            w = w / w.sum()

        (
            ret,
            vol,
            cvar,
            turnover,
            diversification,
            sharpe
        ) = metrics(w)

        # Pymoo minimises all objectives
        out["F"] = np.array([
            -ret,                 # Return ↑
            cvar,                 # CVaR ↓
            turnover,             # Turnover ↓
            -diversification      # Diversification ↑
        ])


# ================================================================
# 4. REFERENCE DIRECTIONS
# ================================================================
# 4 objectives + 5 partitions = 56 subproblems
# This is much faster than 455 directions.
# ================================================================

ref_dirs = get_reference_directions(
    "das-dennis",
    4,
    n_partitions=5
)

print(
    "MOEA/D subproblems:",
    len(ref_dirs)
)


# ================================================================
# 5. MOEA/D ALGORITHM
# ================================================================

problem = PortfolioProblem()

algorithm = MOEAD(
    ref_dirs=ref_dirs,
    n_neighbors=10,
    prob_neighbor_mating=0.7,
    seed=42
)

result = minimize(
    problem,
    algorithm,
    ("n_gen", 150),
    seed=42,
    verbose=False
)


# ================================================================
# 6. PARETO SOLUTIONS
# ================================================================

pareto_weights = result.X

rows = []

valid_weights = []

for w in pareto_weights:

    # Enforce portfolio constraints
    w = np.maximum(
        w,
        0
    )

    if w.sum() == 0:
        continue

    w = w / w.sum()

    (
        ret,
        vol,
        cvar,
        turnover,
        diversification,
        sharpe
    ) = metrics(w)

    valid_weights.append(w)

    rows.append([
        ret,
        vol,
        cvar,
        turnover,
        diversification,
        sharpe
    ])


pareto_weights = np.array(
    valid_weights
)

results = pd.DataFrame(
    rows,
    columns=[
        "Return",
        "Volatility",
        "CVaR",
        "Turnover",
        "Diversification",
        "Sharpe"
    ]
)


# ================================================================
# 7. REMOVE DUPLICATES
# ================================================================

results = results.drop_duplicates(
    subset=[
        "Return",
        "CVaR",
        "Turnover",
        "Diversification"
    ]
).reset_index(drop=True)

pareto_weights = pareto_weights[
    results.index
]


# ================================================================
# 8. BALANCED PORTFOLIO SCORE
# ================================================================

def normalize(x):

    return (
        (x - x.min())
        /
        (
            x.max()
            - x.min()
            + 1e-12
        )
    )


return_score = normalize(
    results["Return"]
)

cvar_score = (
    1 - normalize(
        results["CVaR"]
    )
)

turnover_score = (
    1 - normalize(
        results["Turnover"]
    )
)

diversification_score = normalize(
    results["Diversification"]
)

results["Score"] = (
    return_score
    + cvar_score
    + turnover_score
    + diversification_score
)


# ================================================================
# 9. SELECT BALANCED PORTFOLIO
# ================================================================

best_idx = results[
    "Score"
].idxmax()

best_weights = pareto_weights[
    best_idx
]

best_weights = (
    np.maximum(
        best_weights,
        0
    )
)

best_weights /= best_weights.sum()


# ================================================================
# 10. FINAL PORTFOLIO
# ================================================================

print(
    "\n================================================"
)

print(
    "             MOEA/D OPTIMISED PORTFOLIO"
)

print(
    "================================================\n"
)

for ticker, weight in zip(
    tickers,
    best_weights
):

    print(
        f"{ticker:15s}: {weight:.2%}"
    )


# ================================================================
# 11. PERFORMANCE METRICS
# ================================================================

print(
    "\n================ METRICS =================\n"
)

print(
    results.iloc[best_idx]
    .round(4)
)


# ================================================================
# 12. PARETO FRONTIER
# ================================================================

plt.figure(
    figsize=(10, 6)
)

scatter = plt.scatter(
    results["CVaR"],
    results["Return"],
    c=results["Diversification"],
    s=70
)

plt.scatter(
    results.iloc[best_idx]["CVaR"],
    results.iloc[best_idx]["Return"],
    marker="*",
    s=250,
    label="Selected MOEA/D Portfolio"
)

plt.xlabel(
    "CVaR (95%)"
)

plt.ylabel(
    "Expected Annual Return"
)

plt.title(
    "MOEA/D Pareto Frontier"
)

plt.colorbar(
    scatter,
    label="Diversification"
)

plt.legend()

plt.grid(
    alpha=0.25
)

plt.show()


# ================================================================
# 13. PORTFOLIO WEIGHTS
# ================================================================

plt.figure(
    figsize=(12, 5)
)

plt.bar(
    tickers,
    best_weights
)

plt.xticks(
    rotation=45
)

plt.ylabel(
    "Portfolio Weight"
)

plt.title(
    "MOEA/D Optimised Portfolio Weights"
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.show()


# ================================================================
# 14. PARETO SOLUTION TABLE
# ================================================================

print(
    "\n================ TOP PARETO SOLUTIONS ================\n"
)

print(
    results
    .sort_values(
        "Score",
        ascending=False
    )
    .head(10)
    .round(4)
)

MOEA/D subproblems: 56
